In [ ]:
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Define directories
INPUT_DIR = "output/01-preprocessing"
OUTPUT_DIR = "output/02-shirorekha"

# Create directories if they don't exist
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
image_filename = "preprocessed-0008.png"
input_path = os.path.join(INPUT_DIR, image_filename)

# Read the original image
original_img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)

In [ ]:
# Helper function to display images
def show_image(image, title="Image", cmap_type="gray"):
    plt.figure(figsize=(6, 4))
    plt.imshow(image, cmap=cmap_type)
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
# -------------------- Horizontal Projection Profile --------------------

# Convert to binary (text = 1, background = 0)
binary = (original_img == 0).astype("uint8")

# Row-wise sum
horizontal_profile = np.sum(binary, axis=1)

# Display profile
plt.figure(figsize=(8, 4))
plt.plot(horizontal_profile)
plt.xlabel("Row Index")
plt.ylabel("Dark Pixel Count")
plt.show()

In [ ]:
# -------------------- Headline Threshold Computation --------------------

max_val = np.max(horizontal_profile)
headline_threshold = 0.7 * max_val

# Visualize threshold on graph
plt.figure(figsize=(8, 4))
plt.plot(horizontal_profile, label="Horizontal Profile")
plt.axhline(headline_threshold, linestyle="--", label="Headline Threshold")
plt.title("Profile with Headline Threshold")
plt.xlabel("Row Index")
plt.ylabel("Dark Pixel Count")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# -------------------- Group Rows into Bands --------------------

# Get candidate rows
headline_rows = np.where(horizontal_profile >= headline_threshold)[0]

# Group into contiguous bands
bands = []
current_band = [headline_rows[0]]

for i in range(1, len(headline_rows)):
    if headline_rows[i] == headline_rows[i - 1] + 1:
        current_band.append(headline_rows[i])
    else:
        bands.append(current_band)
        current_band = [headline_rows[i]]

bands.append(current_band)

print(f"Detected bands: {bands}")

In [ ]:
# -------------------- Validate Bands --------------------

h, w = original_img.shape

valid_bands = []

for band in bands:
    coverages = []

    for r in band:
        black_pixels = np.sum(original_img[r] == 0)
        coverage = black_pixels / w
        coverages.append(coverage)

    avg_coverage = np.mean(coverages)

    print(f"Band {band} → Avg coverage: {avg_coverage:.2f}")

    # Keep only wide horizontal structures
    if avg_coverage > 0.6:
        valid_bands.append(band)

print(f"Valid bands: {valid_bands}")

In [ ]:
# -------------------- Shirorekha Visualization --------------------

# Convert grayscale to BGR
shirorekha_detected = cv2.cvtColor(original_img, cv2.COLOR_GRAY2BGR)

# Draw detected shirorekha bands
for band in valid_bands:
    mid_row = int(np.mean(band))
    cv2.line(shirorekha_detected, (0, mid_row), (w - 1, mid_row), (255, 0, 0), 2)

# Show the result
show_image(shirorekha_detected, title="Detected Shirorekha")

# Save image
detected_path = os.path.join(OUTPUT_DIR, "01-detection-" + image_filename)
cv2.imwrite(detected_path, shirorekha_detected)
print(f"Saved shirorekha image at: {detected_path}")

In [ ]:
# -------------------- Shirorekha Removal --------------------

# Copy to preserve original
shirorekha_removed = original_img.copy()

# Remove bands
for band in valid_bands:
    mid_row = int(np.mean(band))
    thickness = 1

    for t in range(-thickness, thickness + 1):
        r = mid_row + t
        if 0 <= r < shirorekha_removed.shape[0]:
            shirorekha_removed[r, :] = 255

# Show the result
show_image(shirorekha_removed, title="Removed Shirorekha")

# Save image
removed_path = os.path.join(OUTPUT_DIR, "02-removal-" + image_filename)
cv2.imwrite(removed_path, shirorekha_removed)
print(f"Shirorekha removed image saved to: {removed_path}")

In [ ]:
# -------------------- Vertical Profile Reconnection --------------------
# Define reconnection zone
cut_rows = [int(np.mean(band)) for band in valid_bands]
zone_top = max(0, min(cut_rows) - 3)
zone_bottom = min(original_img.shape[0], max(cut_rows) + 3)
max_gap = 5

# Convert to binary (text = 1, background = 0)
binary = (shirorekha_removed == 0).astype("uint8")

vertical_profile = np.sum(binary, axis=0)
threshold = 0.75 * np.max(vertical_profile)

candidate_cols = np.where(vertical_profile >= threshold)[0]

# Output image
reconnected = shirorekha_removed.copy()

for col in candidate_cols:
    for r in range(zone_top, zone_bottom):
        if reconnected[r, col] == 0:
            r2 = r + 1

            # Scan downward for next black pixel
            gap = 0
            while r2 < zone_bottom and reconnected[r2, col] == 255:
                gap += 1
                if gap > max_gap:
                    break
                r2 += 1

            # Connect if valid small gap
            if r2 < zone_bottom and gap > 0 and gap <= max_gap:
                for fill_r in range(r + 1, r2):
                    reconnected[fill_r, col] = 0
                r = r2
            else:
                r += 1
        else:
            r += 1

# Show result
show_image(reconnected, title="Reconnected")

reconnected_path = os.path.join(OUTPUT_DIR, "03-reconnected-" + image_filename)
cv2.imwrite(reconnected_path, reconnected)
print(f"Final image saved to: {reconnected_path}")

In [ ]:
# -------------------- Finalise Output Image --------------------

final_path = os.path.join(OUTPUT_DIR, "shirorekha-removed-" + image_filename)
cv2.imwrite(final_path, reconnected)
print(f"Final image saved to: {final_path}")